# 01 Parking rules
What the register contains, what parsed cleanly, and what did not.

Run `python pipeline/process.py` first.

In [ ]:
import geopandas as gpd, pandas as pd, matplotlib.pyplot as plt

areas = gpd.read_parquet("../data/processed/parking_rules.parquet")  # 8,754 areas, metric CRS
print(len(areas), "areas |", areas.crs.to_epsg())

## How much can the app actually answer?

In [ ]:
pd.crosstab(areas["rule_type"], areas["status"], margins=True)

## What made an area uncertain?
Every uncertain area carries the reason. Nothing is guessed.

In [ ]:
import collections
reasons = collections.Counter(
    r.split(":")[0].split(" in ")[0]
    for v in areas["reason"].dropna() for r in v.split("; "))
pd.Series(dict(reasons)).sort_values(ascending=False)

## Raw values behind the parsers
If the register adds a new spelling, it shows up here first.

In [ ]:
for col in ["voimassaolo", "kesto", "kausi"]:
    print(col, "-", areas[col].nunique(), "distinct")
    print(areas[col].value_counts().head(8).to_string(), "\n")

## Where the gaps are
Missing hours cluster outside the city centre.

In [ ]:
ax = areas.plot(column="status", legend=True, figsize=(9, 9), linewidth=2)
ax.set_title("Rule status per parking area"); ax.set_axis_off()

## Roadworks
Areas overlapping a temporary traffic arrangement cannot be trusted today.

In [ ]:
areas["roadworks_until"].notna().sum()